In [9]:
import numpy as np
from zhinst.toolkit import Session
session = Session("localhost", hf2=True)
device = session.connect_device("DEV1004")

In [6]:
device.demods[0].enable()


1

In [7]:


device.demods[0].enable(True)

sample_nodes = [
    device.demods[0].sample.x,
    device.demods[0].sample.y
]



Create and configure the Data Acquisition module
General parameters


In [10]:

TOTAL_DURATION = 5 # [s]
SAMPLING_RATE = 30000 # Number of points/second
BURST_DURATION = 0.2 # Time in seconds for each data burst/segment.

num_cols = int(np.ceil(SAMPLING_RATE * BURST_DURATION))
num_bursts = int(np.ceil(TOTAL_DURATION / BURST_DURATION))



Module creation


In [11]:

daq_module = session.modules.daq
daq_module.device(device)
daq_module.type(0) # continuous acquisition
daq_module.grid.mode(2)
daq_module.count(num_bursts)
daq_module.duration(BURST_DURATION)
daq_module.grid.cols(num_cols)



Configuring the data saving settings


In [12]:

daq_module.save.fileformat(1)
daq_module.save.filename('zi_toolkit_acq_example')
daq_module.save.saveonread(1)



Subscribing to the nodes


In [13]:

for node in sample_nodes:
    daq_module.subscribe(node)

clockbase = device.clockbase()



Recording and plotting the data

Helper function for reading and plotting the data


In [14]:

import matplotlib.pyplot as plt
%matplotlib notebook

def read_and_plot_data(daq_module, results, ts0):
    daq_data = daq_module.read(raw=False, clk_rate=clockbase)
    progress = daq_module.raw_module.progress()[0]
    for node in sample_nodes:
        # Check if node data available
        if node in daq_data.keys():
            for sig_burst in daq_data[node]:
                results[node].append(sig_burst)
                if np.any(np.isnan(ts0)):
                  ts0 = sig_burst.header['createdtimestamp'][0] / clockbase
                # Convert from device ticks to time in seconds.
                t0_burst = sig_burst.header['createdtimestamp'][0] / clockbase
                t = (sig_burst.time + t0_burst) - ts0
                value = sig_burst.value[0, :]
                # Plot the data
                ax1.plot(t, value)
                ax1.set_title(f"Progress of data acquisition: {100 * progress:.2f}%.")
                fig.canvas.draw()
                plt.pause(0.001)
    return results, ts0
    



Execute the measurement


In [16]:

import time

ts0 = np.nan
timeout = 1.5 * TOTAL_DURATION
start_time = time.time()
results = {x: [] for x in sample_nodes}

fig = plt.figure()
ax1 = fig.add_subplot(111)
ax1.set_xlabel("Time ($s$)")
ax1.set_ylabel("Subscribed signals")
ax1.set_xlim([0, TOTAL_DURATION])
ax1.grid()

# Start recording data
daq_module.execute()

while time.time() - start_time < timeout:
    results, ts0 = read_and_plot_data(daq_module, results, ts0)
    if daq_module.raw_module.finished():
        # Once finished, call once more to get the potential remaining data.
        results, ts0 = read_and_plot_data(daq_module, results, ts0)
        break

    time.sleep(BURST_DURATION)



<IPython.core.display.Javascript object>

In [22]:
help(daq_module.grid.mode)

Help on Node in module zhinst.toolkit.nodetree.node:

/grid/mode/__origin


In [26]:
daq_module.grid.mode()

</grid/mode.nearest: 1>

In [17]:
results

{/dev1004/demods/0/sample.x: [DAQResult(header={'systemtime': array([1730159985], dtype=uint64), 'createdtimestamp': array([118851326356016], dtype=uint64), 'changedtimestamp': array([118851326356016], dtype=uint64), 'flags': array([825], dtype=uint32), 'moduleflags': array([0], dtype=uint32), 'chunksizebytes': array([0], dtype=uint64), 'name': '025 19:59:45.3563', 'status': array([0], dtype=uint32), 'groupindex': array([0], dtype=uint32), 'color': array([0], dtype=uint32), 'activerow': array([0], dtype=uint32), 'triggernumber': array([0], dtype=uint64), 'gridrows': array([1], dtype=uint32), 'gridcols': array([6000], dtype=uint32), 'gridmode': array([2], dtype=uint32), 'gridoperation': array([1], dtype=uint32), 'griddirection': array([0], dtype=uint32), 'gridrepetitions': array([1], dtype=uint32), 'gridcoldelta': array([3.33333333e-05]), 'gridcoloffset': array([-0.001]), 'gridrowdelta': array([1.]), 'gridrowoffset': array([0.]), 'bandwidth': array([nan]), 'center': array([0.]), 'nenbw'